# Active Learning Experiment

This notebook compares `entropy` and `random` sampling for the `ActiveLearningAgent`.

Goals:
- load a labeled dataset
- create `initial labeled`, `pool`, and `test` splits
- run 5 AL iterations with `batch_size=20`
- compare learning curves
- estimate sample-efficiency savings

If no project dataset is found, the notebook falls back to a small synthetic text dataset so the experiment stays reproducible.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agents.al_agent import ActiveLearningAgent

DATA_CANDIDATES = [
    PROJECT_ROOT / "data/labeled/final_labeled.parquet",
    PROJECT_ROOT / "data/labeled/auto_labeled.parquet",
    PROJECT_ROOT / "data/interim/cleaned_final.parquet",
    PROJECT_ROOT / "data/interim/cleaned_preview_conservative.parquet",
]
TEXT_COL = "text"
LABEL_COL = "label"
ID_COL = "id"
RANDOM_STATE = 42
INITIAL_N = 50
N_ITERATIONS = 5
BATCH_SIZE = 20


In [ ]:
def build_demo_dataset(n_per_class: int = 120) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for index in range(n_per_class):
        rows.append(
            {
                ID_COL: f"pos_{index}",
                TEXT_COL: f"good movie pleasant story good acting sample {index}",
                LABEL_COL: "positive",
            }
        )
        rows.append(
            {
                ID_COL: f"neg_{index}",
                TEXT_COL: f"bad movie awful story bad acting sample {index}",
                LABEL_COL: "negative",
            }
        )
    return pd.DataFrame(rows)


def load_labeled_dataset() -> pd.DataFrame:
    for candidate in DATA_CANDIDATES:
        if candidate.exists():
            if candidate.suffix == ".csv":
                df = pd.read_csv(candidate)
            else:
                df = pd.read_parquet(candidate)
            display(Markdown(f"Loaded dataset from `{candidate.relative_to(PROJECT_ROOT)}` with {len(df)} rows."))
            return df
    df = build_demo_dataset()
    display(Markdown(f"No project labeled dataset found, using synthetic demo data with {len(df)} rows."))
    return df


df = load_labeled_dataset().copy()
if ID_COL not in df.columns:
    df[ID_COL] = [f"row_{index}" for index in range(len(df))]
if TEXT_COL not in df.columns:
    for candidate in ["prompt", "content", "query"]:
        if candidate in df.columns:
            df = df.rename(columns={candidate: TEXT_COL})
            break
if LABEL_COL not in df.columns:
    for candidate in ["final_label", "auto_label", "target", "class"]:
        if candidate in df.columns:
            df = df.rename(columns={candidate: LABEL_COL})
            break

df = df[[ID_COL, TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)
df.head()

In [ ]:
train_pool_df, test_df = train_test_split(
    df,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=df[LABEL_COL],
)

initial_labeled_df, pool_df = train_test_split(
    train_pool_df,
    train_size=min(INITIAL_N, len(train_pool_df) - 1),
    random_state=RANDOM_STATE,
    stratify=train_pool_df[LABEL_COL],
)

pool_df = pool_df.copy()
pool_df["true_label"] = pool_df[LABEL_COL]
pool_df = pool_df.drop(columns=[LABEL_COL])

display(Markdown(
    f"Initial labeled: **{len(initial_labeled_df)}**  \\n"
    f"Pool: **{len(pool_df)}**  \\n"
    f"Test: **{len(test_df)}**"
))


In [ ]:
agent = ActiveLearningAgent(
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    id_col=ID_COL,
    random_state=RANDOM_STATE,
    output_dir="artifacts/al",
)

entropy_history = agent.run_cycle(
    labeled_df=initial_labeled_df,
    pool_df=pool_df,
    test_df=test_df,
    strategy="entropy",
    n_iterations=N_ITERATIONS,
    batch_size=BATCH_SIZE,
    simulation_mode=True,
    oracle_label_col="true_label",
)

random_history = agent.run_cycle(
    labeled_df=initial_labeled_df,
    pool_df=pool_df,
    test_df=test_df,
    strategy="random",
    n_iterations=N_ITERATIONS,
    batch_size=BATCH_SIZE,
    simulation_mode=True,
    oracle_label_col="true_label",
)

histories = {
    "entropy": entropy_history,
    "random": random_history,
}


In [ ]:
plot_path = agent.report(histories, output_path="reports/learning_curve.png")
summary = agent.compute_sample_efficiency(histories, improved_strategy="entropy", baseline_strategy="random")

entropy_df = pd.DataFrame(entropy_history)
random_df = pd.DataFrame(random_history)
history_df = pd.concat([entropy_df, random_df], ignore_index=True)
history_df.sort_values(["strategy", "iteration"]).reset_index(drop=True)


In [ ]:
if summary["target_reached"]:
    markdown = f"""
## Sample Efficiency Summary

- Target random final F1: **{summary['target_quality']:.4f}**
- Random final labeled size: **{summary['n_labeled_baseline_final']}**
- Entropy labeled size at target: **{summary['n_labeled_improved_at_target']}**
- Saved labeling examples: **{summary['saved_examples']}**
- Learning curve saved to `{Path(plot_path).relative_to(PROJECT_ROOT)}`
"""
else:
    markdown = f"""
## Sample Efficiency Summary

- Entropy did not reach the final random F1 target.
- Saved labeling examples: **0**
- Learning curve saved to `{Path(plot_path).relative_to(PROJECT_ROOT)}`
"""

display(Markdown(markdown))


## Production-like HITL note

For the final pipeline, switch to `simulation_mode=False`. In that mode the agent exports `review_queue_al.csv`, waits for a human to fill `human_label`, and only then adds the reviewed examples back into the training set.